# V2 Feature Extraction

Extracts V2 features from MSR-VTT:
- **Student visual**: 3 CLIP ViT-B/32 frames per video — early (idx 1), middle, late (idx -2) → `z_img` [N, 3, 512]
- **Student audio**: VGGish 128-dim → `z_aud` [N, 128]
- **Teacher**: `normalize(IB_vision + IB_audio) / 2` when audio exists, else `IB_vision` → `v_teacher` [N, 1024]
- **Audio flag**: `has_audio` [N, bool]

Designed to run on Kaggle T4 GPU with **save-and-run-all**. Checkpoints every 200 videos — safe to restart if kernel dies.

## Step 1: Install Dependencies

In [ ]:
!pip install -q git+https://github.com/facebookresearch/ImageBind.git
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q decord pytorchvideo resampy soundfile tqdm

## Step 2: Imports & GPU Check

In [ ]:
import os
import json
import subprocess
import zipfile
import urllib.request

import torch
import torch.nn.functional as F
import numpy as np
import PIL.Image as Image
from tqdm import tqdm
from decord import VideoReader, cpu

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: Running on CPU. Feature extraction will be very slow.")

## Step 3: Download MSR-VTT

In [ ]:
os.makedirs("msrvtt", exist_ok=True)

urls = {
    "msrvtt/msrvtt_train_7k.json": "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/msrvtt_train_7k.json",
    "msrvtt/msrvtt_test_1k.json":  "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/msrvtt_test_1k.json",
    "msrvtt/MSRVTT_Videos.zip":    "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/MSRVTT_Videos.zip",
}

for path, url in urls.items():
    if not os.path.exists(path):
        print(f"Downloading {path}...")
        urllib.request.urlretrieve(url, path)
        print("Done.")
    else:
        print(f"Already exists: {path}")

video_dir = "msrvtt/video"
if not os.path.exists(video_dir):
    print("Extracting videos...")
    with zipfile.ZipFile("msrvtt/MSRVTT_Videos.zip", "r") as zf:
        zf.extractall("msrvtt")
    print("Done.")
else:
    n = len(os.listdir(video_dir))
    print(f"Video dir exists with {n} files.")

## Step 4: Load Encoders

- **CLIP ViT-B/32** — student visual (512-dim)
- **VGGish** — student audio (128-dim)
- **ImageBind huge** — teacher: vision 16-frame + audio (both 1024-dim, same space)

In [ ]:
import clip
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType
import imagebind.data as ib_data

# CLIP
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("CLIP loaded.")

# VGGish
vggish = torch.hub.load("harritaylor/torchvggish", "vggish", trust_repo=True)
vggish.eval().to(device)
print("VGGish loaded.")

# ImageBind
ib_model = imagebind_model.imagebind_huge(pretrained=True)
ib_model.eval().to(device)
print("ImageBind loaded.")

## Step 5: Extraction Helpers

In [ ]:
def check_has_audio(video_path):
    """Return True if the video file has at least one audio stream."""
    cmd = [
        "ffprobe", "-v", "error",
        "-select_streams", "a:0",
        "-show_entries", "stream=codec_name",
        "-of", "default=noprint_wrappers=1",
        video_path,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return bool(result.stdout.strip())


def extract_3_clip_frames(video_path):
    """
    Returns [3, 512] tensor of CLIP embeddings.
    Frames: index 1 (early), middle, index -2 (late).
    For very short videos (<= 2 frames) the three indices may overlap — that is fine.
    """
    vr = VideoReader(video_path, ctx=cpu(0))
    n = len(vr)
    idxs = [min(1, n - 1), n // 2, max(0, n - 2)]
    frame_feats = []
    for idx in idxs:
        frame = vr[idx]
        arr = frame.asnumpy() if hasattr(frame, "asnumpy") else frame.cpu().numpy()
        pil_img = Image.fromarray(arr)
        t = clip_preprocess(pil_img).unsqueeze(0).to(device)
        with torch.no_grad():
            feat = clip_model.encode_image(t).squeeze(0).cpu()  # [512]
        frame_feats.append(feat)
    return torch.stack(frame_feats)  # [3, 512]


def extract_vggish_audio(video_path):
    """Returns [128] VGGish embedding. Returns zeros if audio unavailable."""
    tmp = "/tmp/vggish_audio.wav"
    if os.path.exists(tmp):
        os.remove(tmp)
    cmd = f'ffmpeg -y -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{tmp}"'
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if not os.path.exists(tmp) or os.path.getsize(tmp) < 1000:
        if os.path.exists(tmp):
            os.remove(tmp)
        return torch.zeros(128)
    try:
        with torch.no_grad():
            feat = vggish.forward(tmp)
            if feat.ndim > 1:
                feat = feat.mean(dim=0)
        os.remove(tmp)
        return feat.cpu()
    except Exception:
        if os.path.exists(tmp):
            os.remove(tmp)
        return torch.zeros(128)


def extract_imagebind_vision(video_path):
    """Returns [1024] ImageBind vision embedding (default 16-frame sampling)."""
    inputs = {ModalityType.VISION: ib_data.load_and_transform_video_data([video_path], device)}
    with torch.no_grad():
        emb = ib_model(inputs)[ModalityType.VISION]
    return emb.squeeze(0).cpu()  # [1024]


def extract_imagebind_audio(video_path):
    """
    Returns [1024] ImageBind audio embedding, or None if extraction fails.
    Exports a temp wav and passes it through ImageBind's audio pipeline.
    """
    tmp = "/tmp/ib_audio.wav"
    if os.path.exists(tmp):
        os.remove(tmp)
    cmd = f'ffmpeg -y -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{tmp}"'
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if not os.path.exists(tmp) or os.path.getsize(tmp) < 1000:
        if os.path.exists(tmp):
            os.remove(tmp)
        return None
    try:
        audio_data = ib_data.load_and_transform_audio_data([tmp], device)
        inputs = {ModalityType.AUDIO: audio_data}
        with torch.no_grad():
            emb = ib_model(inputs)[ModalityType.AUDIO]
        os.remove(tmp)
        return emb.squeeze(0).cpu()  # [1024]
    except Exception as e:
        print(f"    IB audio error: {e}")
        if os.path.exists(tmp):
            os.remove(tmp)
        return None


def build_teacher(v_vis, v_aud):
    """
    Both ImageBind vision and audio embeddings live in the same L2-normalized
    1024-dim space. Averaging + renormalizing gives the multimodal centroid.
    Falls back to v_vis if v_aud is None.
    """
    if v_aud is None:
        return v_vis
    combined = (v_vis + v_aud) / 2.0
    return F.normalize(combined, dim=-1)

## Step 6: Extraction Loop with Checkpointing

Checkpoints to `*_ckpt.pt` every 200 videos. If the kernel dies, re-running this cell resumes from the checkpoint automatically.

In [ ]:
CHECKPOINT_EVERY = 200


def run_extraction(split_json, output_pt):
    checkpoint_pt = output_pt.replace(".pt", "_ckpt.pt")

    with open(split_json) as f:
        data = json.load(f)

    # Build unique video id -> filename mapping
    unique_videos = {}
    for item in data:
        unique_videos[item["video_id"]] = item["video"]
    video_ids = sorted(unique_videos.keys())
    print(f"\nTotal unique videos in {split_json}: {len(video_ids)}")

    # --- Resume from checkpoint ---
    if os.path.exists(checkpoint_pt):
        ckpt = torch.load(checkpoint_pt, map_location="cpu")
        z_imgs     = list(ckpt["z_img"])      # list of [3, 512] tensors
        z_auds     = list(ckpt["z_aud"])      # list of [128] tensors
        v_teachers = list(ckpt["v_teacher"])  # list of [1024] tensors
        has_audios = list(ckpt["has_audio"])  # list of bool tensors
        valid_ids  = list(ckpt["video_ids"])
        done_set   = set(valid_ids)
        print(f"Resuming from checkpoint: {len(valid_ids)} videos already processed.")
    else:
        z_imgs, z_auds, v_teachers, has_audios, valid_ids = [], [], [], [], []
        done_set = set()

    remaining = [v for v in video_ids if v not in done_set]
    print(f"Videos left to process: {len(remaining)}")
    if not remaining:
        print("Nothing to do — all videos already in checkpoint.")

    errors = 0
    for i, vid in enumerate(tqdm(remaining, desc=os.path.basename(output_pt))):
        video_file = os.path.join("msrvtt/video", unique_videos[vid])
        if not os.path.exists(video_file):
            errors += 1
            continue
        try:
            # Student features
            z_img = extract_3_clip_frames(video_file)   # [3, 512]
            z_aud = extract_vggish_audio(video_file)    # [128]

            # Teacher: vision (always) + audio (when available)
            has_aud = check_has_audio(video_file)
            v_vis   = extract_imagebind_vision(video_file)  # [1024]
            v_aud   = extract_imagebind_audio(video_file) if has_aud else None
            v_teach = build_teacher(v_vis, v_aud)            # [1024]

            z_imgs.append(z_img)
            z_auds.append(z_aud)
            v_teachers.append(v_teach)
            has_audios.append(torch.tensor(has_aud and v_aud is not None, dtype=torch.bool))
            valid_ids.append(vid)

        except Exception as e:
            print(f"\nError on {vid}: {e}")
            errors += 1
            continue

        # Periodic checkpoint
        if (i + 1) % CHECKPOINT_EVERY == 0:
            _save_dict(checkpoint_pt, z_imgs, z_auds, v_teachers, has_audios, valid_ids)
            n_aud = sum(h.item() for h in has_audios)
            print(f"  Checkpoint @ {len(valid_ids)} videos | audio={n_aud}/{len(valid_ids)} | errors={errors}")

    # Final save
    _save_dict(output_pt, z_imgs, z_auds, v_teachers, has_audios, valid_ids)
    n_aud = sum(h.item() for h in has_audios)
    print(f"\nSaved to {output_pt}")
    print(f"  z_img    : {torch.stack(z_imgs).shape}")
    print(f"  z_aud    : {torch.stack(z_auds).shape}")
    print(f"  v_teacher: {torch.stack(v_teachers).shape}")
    print(f"  has_audio: {n_aud}/{len(valid_ids)} ({100*n_aud/max(len(valid_ids),1):.1f}%) videos have audio")
    print(f"  errors   : {errors}")

    # Clean up checkpoint
    if os.path.exists(checkpoint_pt):
        os.remove(checkpoint_pt)
        print(f"  Checkpoint removed.")


def _save_dict(path, z_imgs, z_auds, v_teachers, has_audios, valid_ids):
    torch.save({
        "z_img":     torch.stack(z_imgs),
        "z_aud":     torch.stack(z_auds),
        "v_teacher": torch.stack(v_teachers),
        "has_audio": torch.stack(has_audios),
        "video_ids": valid_ids,
    }, path)

## Step 7: Run Extraction

Test set first (1000 videos, ~30 min on T4), then train set (7010 videos, ~3.5h on T4).

In [ ]:
run_extraction("msrvtt/msrvtt_test_1k.json",  "test_features_v2.pt")
run_extraction("msrvtt/msrvtt_train_7k.json", "train_features_v2.pt")

## Step 8: Verify and Download

In [ ]:
from IPython.display import FileLink, HTML, display

for fname in ["test_features_v2.pt", "train_features_v2.pt"]:
    if not os.path.exists(fname):
        print(f"MISSING: {fname}")
        continue
    d = torch.load(fname, map_location="cpu")
    n_aud = d["has_audio"].sum().item()
    n_tot = len(d["video_ids"])
    print(f"\n{fname}")
    print(f"  z_img    : {d['z_img'].shape}  (dtype: {d['z_img'].dtype})")
    print(f"  z_aud    : {d['z_aud'].shape}")
    print(f"  v_teacher: {d['v_teacher'].shape}")
    print(f"  has_audio: {n_aud}/{n_tot} ({100*n_aud/n_tot:.1f}%)")
    display(FileLink(fname, result_html_prefix=f"Download {fname}: "))

print("\nAll files in working directory:")
print([f for f in os.listdir(".") if f.endswith(".pt")])